# Notebook 01: Synthetic Data Generation

This notebook demonstrates how the `SyntheticScenarioGenerator` produces fictional,
research-grade dialogue scenarios for the Norm-Constrained Dialogue Framework.

**All scenarios are entirely synthetic.** No real individuals, institutions,
or events are referenced.

### What you will learn
1. How to configure and run the scenario generator
2. The schema structure of a synthetic scenario
3. Distribution of scenario types and respondent profiles
4. How to save the dataset for downstream simulation

In [ ]:
import sys
sys.path.insert(0, "../src")

from norm_dialogue_framework.data.synthetic_generator import SyntheticScenarioGenerator
from norm_dialogue_framework.config import load_config
import pandas as pd
import plotly.express as px

pd.set_option("display.max_colwidth", 80)

## 1. Load configuration

In [ ]:
cfg = load_config("../configs/default.yaml")
print("Random seed:", cfg.project.random_seed)
print("N scenarios:", cfg.data.n_scenarios)
print("Scenario types:", cfg.data.scenario_types)
print("Respondent profiles:", cfg.data.respondent_profiles)

## 2. Generate scenarios

The generator creates scenarios with:
- A structured context summary
- Communication goals aligned with the scenario type
- Risk flags indicating potential sensitivities
- Recommended norms for the dialogue agent
- A latent truth state (visible to the simulator but hidden from the agent)

In [ ]:
gen = SyntheticScenarioGenerator(seed=cfg.project.random_seed)
scenarios = gen.generate(n=50)
df = gen.to_dataframe(scenarios)
print(f"Generated {len(scenarios)} scenarios")
df.head()

## 3. Inspect a single scenario

In [ ]:
s = scenarios[0]
print("Case ID:", s.case_id)
print("Scenario Type:", s.scenario_type)
print("Respondent Profile:", s.respondent_profile)
print("Sensitivity Level:", s.sensitivity_level)
print()
print("Context Summary:")
print(" ", s.context_summary)
print()
print("Communication Goals:", s.communication_goals)
print("Risk Flags:", s.risk_flags)
print("Recommended Norms:", s.recommended_norms)
print()
print("Latent Truth State (hidden from agent):")
for k, v in s.latent_truth_state.items():
    print(f"  {k}: {v}")

## 4. Explore distributions

In [ ]:
fig = px.bar(
    df["scenario_type"].value_counts().reset_index(),
    x="scenario_type", y="count",
    title="Scenario Type Distribution",
    color="scenario_type"
)
fig.show()

In [ ]:
fig2 = px.sunburst(
    df, path=["scenario_type", "respondent_profile"],
    title="Scenario Type x Respondent Profile Distribution"
)
fig2.show()

In [ ]:
fig3 = px.histogram(
    df, x="sensitivity_level",
    color="scenario_type",
    barmode="group",
    title="Sensitivity Level by Scenario Type"
)
fig3.show()

## 5. Save dataset

In [ ]:
import pathlib
pathlib.Path("../data/synthetic").mkdir(parents=True, exist_ok=True)
gen.save(scenarios, output_dir="../data/synthetic")
print("Dataset saved to data/synthetic/")
print("Files: scenarios.json, scenarios.csv")